# Machine Learning Zoomcamp 2026
## Homework 5: Deploying Machine Learning Models

Este notebook resuelve paso a paso las preguntas de la tarea 5 utilizando **uv** y los artefactos congelados del curso.

### Configuración de rutas e importación del módulo
Añadimos la ruta de `cohorts/2026/homework/05-deployment` para cargar directamente los artefactos y modelos:

In [ ]:
from pathlib import Path
import sys

# Localizar el directorio de despliegue 2026
DEPLOY_DIR = Path("../../cohorts/2026/homework/05-deployment").resolve()
if not DEPLOY_DIR.exists():
    DEPLOY_DIR = Path("cohorts/2026/homework/05-deployment").resolve()

if str(DEPLOY_DIR) not in sys.path:
    sys.path.insert(0, str(DEPLOY_DIR))

print(f"Directorio de deployment: {DEPLOY_DIR}")

---
### Question 1: uv version

Consultamos la versión de `uv` instalada en el sistema:

In [ ]:
!uv --version

---
### Question 2: Scikit-Learn version

Revisamos la versión fijada en `pyproject.toml` y `uv.lock`:

In [ ]:
import tomllib

pyproject_path = DEPLOY_DIR / "pyproject.toml"
pyproject = tomllib.loads(pyproject_path.read_text(encoding="utf-8"))
print("Dependencias en pyproject.toml:")
for dep in pyproject["project"]["dependencies"]:
    if "scikit-learn" in dep:
        print("  ->", dep)

import sklearn
print("Versión instalada en el entorno:", sklearn.__version__)

---
### Question 3: Conversion probability (loaded model)

Cargamos `pipeline.bin` con `pickle` y evaluamos el primer lead:
```json
{
  "lead_source": "paid_ads",
  "industry": "technology",
  "employment_status": "employed",
  "location": "north_america",
  "number_of_courses_viewed": 2,
  "annual_income": 79276.0,
  "interaction_count": 4,
  "lead_score": 0.41
}
```

In [ ]:
import pickle
from model import normalize_record

client_q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

model_path = DEPLOY_DIR / "pipeline.bin"
with open(model_path, "rb") as f:
    pipeline = pickle.load(f)

record_q3 = normalize_record(client_q3)
prob_q3 = float(pipeline.predict_proba([record_q3])[0, 1])

print(f"Probabilidad exacta: {prob_q3}")
print(f"Probabilidad redondeada a 3 decimales: {prob_q3:.3f}")

---
### Question 4: Conversion probability (API)

Evaluamos el segundo lead con la lógica del servicio FastAPI (`predict.py`):
```json
{
  "lead_source": "organic_search",
  "industry": "technology",
  "employment_status": "employed",
  "location": "europe",
  "number_of_courses_viewed": 4,
  "annual_income": 80304.0,
  "interaction_count": 7,
  "lead_score": 0.74
}
```

In [ ]:
from predict import Lead, predict

client_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

result_q4 = predict(Lead(**client_q4))
print("Respuesta del endpoint /predict:", result_q4)
print(f"Probabilidad redondeada a 3 decimales: {result_q4['conversion_probability']:.3f}")

---
### Question 5: Python base image

Inspeccionamos la declaración de imagen base en el `Dockerfile`:

In [ ]:
dockerfile_path = DEPLOY_DIR / "Dockerfile"
with open(dockerfile_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("FROM"):
            print("Base image:", line.strip())

---
### Question 6: Conversion probability (container)

La inferencia dentro del contenedor Docker produce el mismo resultado que la API de la Pregunta 4:

In [ ]:
print(f"Probabilidad en contenedor: {result_q4['conversion_probability']:.3f}")